### Intraday 15 min brakout or breakdown strategy

In [157]:
import os
import sys
from datetime import datetime, timedelta
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


In [183]:
from brokers.fyers.login import Login
import pandas as pd
import pandas_ta as pta
import numpy as np
import ta
login = Login()

fyers = login.get()

---Removing all previous day file----


In [252]:
def fetch_stock_data(data):
    response = fyers.history(data=data)
    
    if response['code'] != 200:
        raise Exception(f"{response['message']}")
    if len(response["candles"]) == 0:
        raise Exception(f"Holiday on {data['range_from']}")
    df = pd.DataFrame(columns=["date","open","high","low","close","volumn"], data=response["candles"])
    df['date'] = pd.to_datetime(df['date'],unit='s')
    df['date'] = df['date'].dt.tz_localize('utc').dt.tz_convert('Asia/kolkata')
    df['date'] = df['date'].dt.tz_localize(None)
    return df

In [174]:
def calculate_levels(data, lookback_period=60):
    data['High_15m'] = max([data["high"].iloc[0]], [data["high"].iloc[1]], [data["high"].iloc[3]])[0]
    data['Low_15m'] = min([data["low"].iloc[0]], [data["low"].iloc[1]],[data["low"].iloc[2]])[0]
    # Rolling for candle
    #  = data['high'].rolling(window=lookback_period, min_periods=1).max()
    #  = data['low'].rolling(window=lookback_period, min_periods=1).min()
    return data

In [262]:
def generate_signals(data):
    data['Signal'] = 0
    data['Signal'][data['close'] > data['High_15m']] = 1  # Buy signal
    data['Signal'][data['close'] < data['Low_15m']] = -1  # Sell signal
    return data

In [ ]:
def take_entry

In [270]:
def apply_trailing_stop(data, stop_loss_percent):
    pnl = {
        
    }
    flag = False
    for i in range(len(data)):
        if data.loc[i,'Signal'] !=0:
            flag =True 
        if flag == True:
            if data.loc[i,'Signal'] == 1:
                print("buy")
            elif data.loc[i,'Signal'] == -1:
                print("Sell")

In [264]:
import time
def start(start_date, days):
    end_date = start_date + timedelta(days=days)
    while start_date < end_date:
        try:
            date = start_date.strftime('%Y-%m-%d')
            data = fetch_stock_data(data = {
                "symbol":"NSE:SBIN-EQ",
                "resolution":"5",
                "date_format":"1",
                "range_from":date,
                "range_to":date,
                "cont_flag":"1"

            })
            data = calculate_levels(data, lookback_period=74)
            # data = generate_signals(data)
            data = apply_trailing_stop(data=data,stop_loss_percent=.5/100)
            # data.to_csv("SBIN.csv", mode='a', header=True)
        
        except Exception as e:
            print(e)
        time.sleep(5)
        start_date += timedelta(days=1)
        

In [271]:
pnl_list = start(start_date = datetime(2024, 7, 29), days=1)
df = pd.DataFrame(pnl_list)
pnl_list
# df.to_csv('SBI.csv', index=False, header=False)

/var/folders/f9/rddmp2b15217gynyrl4qjwrc0000gn/T/ipykernel_7121/3990048602.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data['Signal'][data['close'] > data['High_15m']] = 1  # Buy signal
/var/folders/f9/rddmp2b15217gynyrl4qjwrc0000gn/

buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
buy
'NoneType' object is not subscriptable


In [146]:
df = pd.read_csv("SBI.csv")
df

,2024-07-29 12:15:00,Buy,SL,6.9500000000000455,0,882.0,892.584,878.472,0.004,0.012
0,2024-07-30 13:00:00,Sell,SL,4.35,0,870.6,860.1528,874.0824,0.004,0.012
